# Projeto 2 — Regressão Linear para a nota de Linguagens (`NU_NOTA_LC`) — ENEM 2023

Este notebook treina **somente a família de Regressão Linear** sobre a base já tratada
(`dados/processed/enem_2023_lc_elegiveis.parquet`, 2.822.643 participantes com nota de LC):

| Modelo | Papel |
|---|---|
| `DummyRegressor(strategy="mean")` | **Baseline** obrigatório: prevê a média do treino para todos |
| `LinearRegression()` | Regressão linear **sem** regularização (MQO) |
| `Ridge(alpha)` | Regressão linear **com** regularização L2 (+ sensibilidade ao `alpha`) |

**Protocolo (Seção 4 do enunciado)**

1. Divisão 80% desenvolvimento / 20% teste com `random_state=42`; identificadores salvos em `dados/splits/`.
2. No desenvolvimento, `KFold(n_splits=3, shuffle=True, random_state=42)`; as mesmas dobras em todas as comparações.
3. Toda a preparação (codificação, imputação, padronização) fica dentro de um `Pipeline`, então é reajustada **só no treino de cada dobra**.
4. Métrica principal RMSE; complementares MAE e R²; média e desvio padrão nas 3 dobras; tempos registrados.
5. **O teste não é avaliado neste notebook** (o enunciado proíbe avaliar o teste no acompanhamento e antes de fixar a solução final). A última seção já deixa o código pronto, desligado por `AVALIAR_TESTE = False`.

**Como rodar no Colab:** envie `enem_2023_lc_elegiveis.parquet` pelo painel de arquivos (ícone de pasta, à esquerda) e use
`Ambiente de execução → Executar tudo`. Se o arquivo não tiver sido enviado, a Seção 2 abre um botão de upload.
No fim, o notebook baixa um `.zip` com todas as saídas. Tudo roda em CPU; o tempo total esperado é de alguns minutos.

## 1. Ambiente e sementes

In [ ]:
import json
import re
import time
import platform
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import (GridSearchCV, KFold, cross_val_predict, cross_validate,
                                     learning_curve, train_test_split)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler

SEMENTE = 42
np.random.seed(SEMENTE)

print("Python      ", platform.python_version())
print("numpy       ", np.__version__)
print("pandas      ", pd.__version__)
print("scikit-learn", sklearn.__version__)
print("matplotlib  ", matplotlib.__version__)

In [ ]:
# Estilo dos gráficos: marcas finas, grade discreta, cores fixas por papel.
COR_MODELO = "#2a78d6"      # validação / modelo avaliado
COR_TREINO = "#eb6834"      # treino
COR_BASELINE = "#8a8984"    # baseline e referências neutras
COR_TEXTO_SEC = "#52514e"

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e4e3df", "grid.linewidth": 0.8,
    "axes.edgecolor": "#b5b4ae", "axes.titleweight": "bold", "axes.titlesize": 12,
    "axes.labelcolor": COR_TEXTO_SEC, "xtick.color": COR_TEXTO_SEC, "ytick.color": COR_TEXTO_SEC,
    "legend.frameon": False, "font.size": 10,
})
FONTE_DADOS = "Fonte: Inep, Microdados ENEM 2023. Elaboração da equipe."

## 2. Localização dos arquivos

No Colab, o notebook monta a estrutura do projeto em `/content/AV2_CienciaDados` e procura o arquivo
**`enem_2023_lc_elegiveis.parquet`** (≈ 31 MB) que você enviou pelo painel de arquivos (em qualquer pasta de `/content`).
Se não encontrar, abre um botão **"Escolher arquivos"** para o upload.

O Colab apaga `/content` quando a sessão desconecta: por isso a última célula baixa um `.zip` com as saídas
(partições, `experimentos.csv`, tabelas e figuras) para você copiar para o repositório.

Fora do Colab (Jupyter local), o notebook usa a pasta acima de `notebooks/` como raiz do projeto.

In [ ]:
NOME_BASE = "enem_2023_lc_elegiveis.parquet"
ARQUIVO_BASE = Path("dados/processed") / NOME_BASE

try:
    from google.colab import files
    NO_COLAB = True
except ImportError:
    NO_COLAB = False

if NO_COLAB:
    PASTA_PROJETO = Path("/content/AV2_CienciaDados")
    destino = PASTA_PROJETO / ARQUIVO_BASE
    destino.parent.mkdir(parents=True, exist_ok=True)
    if not destino.exists():
        enviados = [p for p in Path("/content").rglob(NOME_BASE) if p != destino]
        if enviados:
            enviados[0].replace(destino)            # arquivo enviado pelo painel lateral
        else:
            print(f"Envie o arquivo {NOME_BASE}:")
            for nome, conteudo in files.upload().items():
                destino.write_bytes(conteudo)
else:
    PASTA_PROJETO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PASTA_SPLITS = PASTA_PROJETO / "dados" / "splits"
PASTA_RESULTADOS = PASTA_PROJETO / "resultados"
PASTA_FIGURAS = PASTA_RESULTADOS / "figuras"
ARQ_PARTICOES = PASTA_SPLITS / "particoes_lc.parquet"
ARQ_EXPERIMENTOS = PASTA_RESULTADOS / "experimentos.csv"
ARQ_PREVISOES_TESTE = PASTA_RESULTADOS / "previsoes_teste.csv"
for pasta in (PASTA_SPLITS, PASTA_RESULTADOS, PASTA_FIGURAS):
    pasta.mkdir(parents=True, exist_ok=True)

print("Projeto:", PASTA_PROJETO.resolve())
print("Base   :", (PASTA_PROJETO / ARQUIVO_BASE).exists())

## 3. Carga e conferência da base tratada

Conferências antes de qualquer modelagem: número de linhas igual ao do tratamento, identificador único,
alvo sempre observado e **nenhuma coluna proibida** (outras notas, respostas, gabaritos, presença, situação da redação, código de prova).

In [ ]:
COLUNA_ID = "NU_INSCRICAO"
ALVO = "NU_NOTA_LC"
TOTAL_INSCRITOS = 3_933_955          # MICRODADOS_ENEM_2023.csv completo (diagnóstico)
TOTAL_ELEGIVEIS_ESPERADO = 2_822_643 # participantes com NU_NOTA_LC observada (inclui nota zero)

df = pd.read_parquet(PASTA_PROJETO / ARQUIVO_BASE)
print("Formato:", df.shape)

assert len(df) == TOTAL_ELEGIVEIS_ESPERADO, "A base não tem o total esperado de elegíveis: confira o arquivo."
assert df[COLUNA_ID].is_unique, "Há inscrições duplicadas."
assert df[ALVO].notna().all(), "Há nota alvo ausente na base de elegíveis."
padrao_proibido = r"^(NU_NOTA_(?!LC$)|TX_RESPOSTAS_|TX_GABARITO_|TP_PRESENCA_|TP_STATUS_REDACAO|CO_PROVA_)"
assert not [c for c in df.columns if re.match(padrao_proibido, c)], "Coluna com risco de vazamento na base."
print("Conferências OK.")

df.dtypes.to_frame("tipo").T

## 4. Divisão desenvolvimento/teste e dobras

* 80% desenvolvimento / 20% teste, `train_test_split(..., test_size=0.20, random_state=42)`.
* Dobras: `KFold(n_splits=3, shuffle=True, random_state=42)` sobre o desenvolvimento.
* O resultado é salvo em `dados/splits/particoes_lc.parquet` com as colunas `NU_INSCRICAO`, `CONJUNTO`,
  `DOBRA` (1–3 no desenvolvimento, 0 no teste) e `ORDEM` (posição da linha dentro do conjunto).
* **Se o arquivo já existir, ele é reutilizado** — assim os notebooks dos outros modelos usam exatamente
  os mesmos participantes e as mesmas dobras.

In [ ]:
if ARQ_PARTICOES.exists():
    particoes = pd.read_parquet(ARQ_PARTICOES)
    print("Partições existentes carregadas de", ARQ_PARTICOES)
else:
    ids_dev, ids_teste = train_test_split(df[COLUNA_ID], test_size=0.20, random_state=SEMENTE, shuffle=True)
    dobra = np.zeros(len(ids_dev), dtype="int8")
    kfold = KFold(n_splits=3, shuffle=True, random_state=SEMENTE)
    for k, (_, idx_validacao) in enumerate(kfold.split(ids_dev), start=1):
        dobra[idx_validacao] = k
    particoes = pd.concat([
        pd.DataFrame({COLUNA_ID: ids_dev.to_numpy(), "CONJUNTO": "desenvolvimento",
                      "DOBRA": dobra, "ORDEM": np.arange(len(ids_dev))}),
        pd.DataFrame({COLUNA_ID: ids_teste.to_numpy(), "CONJUNTO": "teste",
                      "DOBRA": np.int8(0), "ORDEM": np.arange(len(ids_teste))}),
    ], ignore_index=True)
    particoes.to_parquet(ARQ_PARTICOES, index=False)
    print("Partições criadas e salvas em", ARQ_PARTICOES)

assert particoes[COLUNA_ID].is_unique and len(particoes) == len(df)

base = df.merge(particoes, on=COLUNA_ID, how="inner", validate="one_to_one")
dev = (base[base["CONJUNTO"] == "desenvolvimento"].sort_values("ORDEM").reset_index(drop=True))
teste = (base[base["CONJUNTO"] == "teste"].sort_values("ORDEM").reset_index(drop=True))
del df, base

y_dev = dev[ALVO].to_numpy()
# As dobras são reconstruídas a partir da coluna salva: (índices de treino, índices de validação).
DOBRAS = [(np.flatnonzero(dev["DOBRA"].to_numpy() != k), np.flatnonzero(dev["DOBRA"].to_numpy() == k))
          for k in (1, 2, 3)]

contagem = pd.DataFrame({
    "etapa": ["Inscritos no arquivo nacional", "Sem nota de LC (fora da regressão)", "Elegíveis (com NU_NOTA_LC)",
              "  Desenvolvimento (80%)", "  Teste reservado (20%)"],
    "participantes": [TOTAL_INSCRITOS, TOTAL_INSCRITOS - TOTAL_ELEGIVEIS_ESPERADO, len(dev) + len(teste),
                      len(dev), len(teste)],
})
display(contagem.style.format({"participantes": "{:,.0f}"}).hide(axis="index"))
print("Tamanho das dobras de validação:", [len(v) for _, v in DOBRAS])
print("Notas zero no desenvolvimento:", int((y_dev == 0).sum()))

## 5. Exploração (somente desenvolvimento)

Dois gráficos que orientam a modelagem linear: a distribuição do alvo (com as notas zero destacadas)
e a nota média por categoria das principais variáveis. Se a média cresce de forma aproximadamente
monotônica com a categoria, faz sentido usar a **posição ordinal** como número; se não, usamos **one-hot**.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(y_dev[y_dev > 0], bins=np.arange(0, 840, 10), color=COR_MODELO, edgecolor="white", linewidth=0.4)
n_zeros = int((y_dev == 0).sum())
ax.bar(0, n_zeros, width=10, align="edge", color=COR_TREINO, edgecolor="white", linewidth=0.4)
ax.annotate(f"nota zero: {n_zeros:,} participantes".replace(",", "."), xy=(5, n_zeros), xytext=(0.06, 0.55),
            textcoords="axes fraction", arrowprops=dict(arrowstyle="-", color=COR_TEXTO_SEC, lw=0.8), color=COR_TEXTO_SEC)
ax.axvline(y_dev.mean(), color=COR_BASELINE, lw=1.5, ls="--")
ax.text(y_dev.mean() + 6, ax.get_ylim()[1] * 0.92, f"média = {y_dev.mean():.1f}", color=COR_TEXTO_SEC)
ax.set(title="Distribuição de NU_NOTA_LC — desenvolvimento (n = %s)" % f"{len(y_dev):,}".replace(",", "."),
       xlabel="Nota de Linguagens", ylabel="Participantes")
ax.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:,.0f}".replace(",", ".")))
fig.text(0, -0.04, FONTE_DADOS, fontsize=8, color=COR_TEXTO_SEC)
fig.savefig(PASTA_FIGURAS / "lc_dist_alvo_dev.png")
plt.show()

print(pd.Series(y_dev).describe().round(1).to_string())

In [ ]:
painel = [("Q006", "Renda familiar (Q006)"), ("ESCOLARIDADE_MAX_RESP", "Escolaridade máx. dos responsáveis"),
          ("TP_FAIXA_ETARIA", "Faixa etária (TP_FAIXA_ETARIA)"), ("Q024", "Computadores em casa (Q024)")]
fig, eixos = plt.subplots(1, 4, figsize=(16, 3.8), sharey=True)
for ax, (coluna, titulo) in zip(eixos, painel):
    resumo = dev.groupby(coluna, observed=True)[ALVO].agg(["mean", "size"]).sort_index()
    rotulos = [str(i).replace("INDETERMINADA", "Indet.") for i in resumo.index]
    ax.plot(range(len(resumo)), resumo["mean"], color=COR_MODELO, lw=2, marker="o", ms=5)
    ax.axhline(y_dev.mean(), color=COR_BASELINE, lw=1, ls="--")
    ax.set_xticks(range(len(resumo)), rotulos, fontsize=8, rotation=90 if len(resumo) > 10 else 0)
    ax.set_title(titulo, fontsize=10)
eixos[0].set_ylabel("Nota média de LC")
fig.suptitle("Nota média por categoria — desenvolvimento (linha tracejada = média geral)", fontweight="bold", y=1.03)
fig.text(0, -0.08, FONTE_DADOS, fontsize=8, color=COR_TEXTO_SEC)
fig.savefig(PASTA_FIGURAS / "lc_media_por_categoria_dev.png")
plt.show()

**Leitura esperada:** renda (Q006), escolaridade e computadores crescem de forma monotônica com a nota → codificação
ordinal (A=0, B=1, …) é adequada para um modelo linear. A faixa etária **não** é monotônica (estável até ~30 anos e cai depois),
e os códigos não têm espaçamento uniforme (1 ano até o código 10, depois 5 anos) → **one-hot**.
`INDETERMINADA`/`H` ("não sei") não tem posição na ordem → recebe um **indicador próprio**.

## 6. Features e redundâncias exatas

A base tratada traz 12 features originais + 3 derivadas. Algumas são **combinações exatas (ou quase)** de outras,
o que deixa a matriz de um modelo linear com **posto incompleto** ou quase singular — os coeficientes deixam de ser identificáveis
(as previsões continuam válidas, mas a interpretação dos coeficientes não):

| Feature | Redundante com | Decisão para os modelos lineares |
|---|---|---|
| `IN_TREINEIRO` | `TP_ST_CONCLUSAO == 3` (cruzamento perfeito na base) | Sai de `X`; a informação continua no one-hot de `TP_ST_CONCLUSAO`. **Nenhum participante é removido.** |
| `Q025` (internet) | `PERFIL_ACESSO_DIGITAL` (é uma das duas variáveis que formam o perfil) | Sai de `X` quando o perfil está presente |
| `TP_ESCOLA = 1` ("não respondeu") | `TP_ST_CONCLUSAO ≠ 2`: o Inep só coleta o tipo de escola de quem conclui o EM em 2023 (na base completa, só 5 exceções em 2,8 milhões) | `TP_ESCOLA` entra só como indicador **escola privada** (`TP_ESCOLA = 3`); "não respondeu" já está representado por `TP_ST_CONCLUSAO` |

A célula abaixo comprova as três redundâncias no desenvolvimento.

In [ ]:
FEATURES_ORIGINAIS = ["Q006", "Q001", "Q002", "TP_ESCOLA", "TP_ST_CONCLUSAO", "TP_FAIXA_ETARIA",
                      "Q005", "Q024", "Q025", "IN_TREINEIRO", "SG_UF_PROVA", "TP_LINGUA"]
FEATURES_DERIVADAS = ["RENDA_PER_CAPITA_APROX", "ESCOLARIDADE_MAX_RESP", "PERFIL_ACESSO_DIGITAL"]

# Conjuntos usados nos modelos lineares
X_ORIGINAIS = [c for c in FEATURES_ORIGINAIS if c != "IN_TREINEIRO"]
X_COMPLETO = [c for c in FEATURES_ORIGINAIS + FEATURES_DERIVADAS if c not in ("IN_TREINEIRO", "Q025")]

display(pd.crosstab(dev["TP_ST_CONCLUSAO"], dev["IN_TREINEIRO"]))
display(pd.crosstab(dev["PERFIL_ACESSO_DIGITAL"], dev["Q025"]))
display(pd.crosstab(dev["TP_ST_CONCLUSAO"], dev["TP_ESCOLA"]))
print("X_ORIGINAIS:", len(X_ORIGINAIS), "colunas ->", X_ORIGINAIS)
print("X_COMPLETO :", len(X_COMPLETO), "colunas ->", X_COMPLETO)

## 7. Pré-processamento (dentro do `Pipeline`)

As categorias são fixadas pelo **dicionário do Inep** (não aprendidas dos dados), então nenhuma dobra encontra categoria desconhecida.
Mediana de imputação e média/desvio da padronização são aprendidas **só no treino de cada dobra**.

| Bloco | Colunas | Transformação | Justificativa |
|---|---|---|---|
| Renda (faixa) | `Q006` | ordinal A–Q → 0–16, padronizada | faixas ordenadas; média cresce monotonicamente |
| Computadores | `Q024` | ordinal A–E → 0–4, padronizada | contagem limitada (E = "4 ou mais") |
| Internet | `Q025` | A→0, B→1 | binária (só no conjunto de originais) |
| Escolaridade | `Q001`, `Q002`, `ESCOLARIDADE_MAX_RESP` | A–G → 0–6; `H`/`INDETERMINADA` → ausente → mediana do treino **+ indicador**; padronizada | "não sei" não tem posição na escala; o indicador faz o modelo estimar um efeito próprio para ele (em modelo linear, com indicador, o valor imputado só desloca o coeficiente do indicador) |
| Renda per capita | `RENDA_PER_CAPITA_APROX` | `log1p` → mediana do treino **+ indicador** → padronizada | assimétrica (log); vazia apenas na faixa aberta `Q006 = Q`, que o indicador identifica |
| Moradores | `Q005` | padronizada | contagem 1–20 |
| Binárias | `TP_LINGUA` (`IN_TREINEIRO` só no diagnóstico de posto) | sem alteração | já são 0/1 |
| Escola privada | `TP_ESCOLA` | 1 se privada (3), 0 se pública (2) ou não respondeu (1) | ver Seção 6: "não respondeu" é quase idêntico a não concluir em 2023 |
| One-hot | `SG_UF_PROVA`, `TP_ST_CONCLUSAO`, `TP_FAIXA_ETARIA`, `PERFIL_ACESSO_DIGITAL` | one-hot sem a categoria de referência | categorias sem ordem (ou ordem não linear); a referência evita colinearidade com o intercepto |

Categorias de referência (excluídas do one-hot via `drop=[...]`): UF **SP**, **já concluiu** o EM, faixa etária **< 17 anos**, **sem computador e sem internet**.

In [ ]:
ORDEM_RENDA = list("ABCDEFGHIJKLMNOPQ")
ORDEM_COMPUTADORES = list("ABCDE")
NIVEL_ESCOLARIDADE = {letra: nivel for nivel, letra in enumerate("ABCDEFG")}  # 'H' e 'INDETERMINADA' ficam ausentes

UFS = ["AC", "AL", "AM", "AP", "BA", "CE", "DF", "ES", "GO", "MA", "MG", "MS", "MT", "PA",
       "PB", "PE", "PI", "PR", "RJ", "RN", "RO", "RR", "RS", "SC", "SE", "SP", "TO"]
# coluna: (categorias do dicionário em ordem crescente, categoria de referência que sai do one-hot)
CATEGORIAS_ONE_HOT = {
    "SG_UF_PROVA": (UFS, "SP"),
    "TP_ST_CONCLUSAO": ([1, 2, 3, 4], 1),
    "TP_FAIXA_ETARIA": (list(range(1, 21)), 1),
    "PERFIL_ACESSO_DIGITAL": (["COMPUTADOR_E_INTERNET", "SEM_COMPUTADOR_SEM_INTERNET",
                               "SOMENTE_COMPUTADOR", "SOMENTE_INTERNET"], "SEM_COMPUTADOR_SEM_INTERNET"),
}


def escolaridade_para_nivel(X):
    """Converte A–G em 0–6; 'H' (não sei) e 'INDETERMINADA' viram NaN para receber indicador próprio."""
    return pd.DataFrame(X).apply(lambda coluna: coluna.map(NIVEL_ESCOLARIDADE)).astype(float)


def escola_privada(X):
    """1 se TP_ESCOLA = 3 (privada); 0 se pública (2) ou não respondeu (1)."""
    return (pd.DataFrame(X) == 3).astype(float)


def montar_preprocessador(colunas):
    """ColumnTransformer com os blocos da tabela acima, restrito às colunas pedidas."""
    blocos = [
        ("renda_faixa", make_pipeline(OrdinalEncoder(categories=[ORDEM_RENDA]), StandardScaler()), ["Q006"]),
        ("computadores", make_pipeline(OrdinalEncoder(categories=[ORDEM_COMPUTADORES]), StandardScaler()), ["Q024"]),
        ("internet", OrdinalEncoder(categories=[["A", "B"]]), ["Q025"]),
        ("escolaridade", make_pipeline(
            FunctionTransformer(escolaridade_para_nivel, feature_names_out="one-to-one"),
            SimpleImputer(strategy="median", add_indicator=True), StandardScaler()),
         ["Q001", "Q002", "ESCOLARIDADE_MAX_RESP"]),
        ("renda_per_capita", make_pipeline(
            FunctionTransformer(np.log1p, feature_names_out="one-to-one"),
            SimpleImputer(strategy="median", add_indicator=True), StandardScaler()),
         ["RENDA_PER_CAPITA_APROX"]),
        ("moradores", StandardScaler(), ["Q005"]),
        ("binarias", "passthrough", ["TP_LINGUA", "IN_TREINEIRO"]),
        ("escola_privada", FunctionTransformer(escola_privada, feature_names_out="one-to-one"), ["TP_ESCOLA"]),
    ]
    transformadores = [(nome, transf, [c for c in cols if c in colunas]) for nome, transf, cols in blocos]
    transformadores = [t for t in transformadores if t[2]]
    one_hot = [c for c in CATEGORIAS_ONE_HOT if c in colunas]
    if one_hot:
        transformadores.append(("one_hot", OneHotEncoder(categories=[CATEGORIAS_ONE_HOT[c][0] for c in one_hot],
                                                         drop=[CATEGORIAS_ONE_HOT[c][1] for c in one_hot],
                                                         sparse_output=False), one_hot))
    usadas = {c for _, _, cols in transformadores for c in cols}
    assert usadas == set(colunas), f"Colunas sem bloco de pré-processamento: {set(colunas) - usadas}"
    return ColumnTransformer(transformadores, remainder="drop", sparse_threshold=0)


def montar_pipeline(colunas, regressor):
    return Pipeline([("preparacao", montar_preprocessador(colunas)), ("regressor", regressor)])

### 7.1 Diagnóstico de posto da matriz

Verificação numérica da Seção 6: posto da matriz de features (mais o intercepto) **com** e **sem** as duas colunas redundantes.
O pré-processador é ajustado no desenvolvimento só para este diagnóstico; os modelos avaliados abaixo reajustam tudo dentro de cada dobra.

In [ ]:
def posto_da_matriz(colunas):
    Xt = montar_preprocessador(colunas).fit_transform(dev[colunas])
    Xt = np.column_stack([np.ones(len(Xt)), Xt])
    gram = Xt.T @ Xt                       # (p × p): mesmo posto de Xt, muito mais barato que SVD de n × p
    return Xt.shape[1], np.linalg.matrix_rank(gram)

for nome, colunas in [("15 features (com IN_TREINEIRO e Q025)", FEATURES_ORIGINAIS + FEATURES_DERIVADAS),
                      ("X_COMPLETO (sem as redundantes)", X_COMPLETO),
                      ("X_ORIGINAIS", X_ORIGINAIS)]:
    p, posto = posto_da_matriz(colunas)
    print(f"{nome:40s} colunas = {p:3d}  posto = {posto:3d}  {'OK' if p == posto else '<- deficiente'}")

## 8. Registro de experimentos

Cada experimento vira uma linha em `resultados/experimentos.csv` (o mesmo arquivo pode receber os experimentos
dos outros modelos; linhas com o mesmo `id_experimento` são substituídas ao rodar de novo).
Campos: hipótese, features, preparação, modelo, parâmetros, conjunto avaliado, métricas por dobra, média, desvio padrão
(amostral, `ddof=1`), RMSE de treino, tempos e conclusão.

In [ ]:
METRICAS = {"rmse": "neg_root_mean_squared_error", "mae": "neg_mean_absolute_error", "r2": "r2"}
PREPARACAO_PADRAO = ("ordinal Q006/Q024; escolaridade A-G + indicador 'não sei'; log1p renda per capita + "
                     "indicador; padronização; escola privada 0/1; one-hot UF/conclusão/faixa etária/perfil digital (sem a categoria de referência)")


def salvar_experimento(registro):
    tabela = pd.read_csv(ARQ_EXPERIMENTOS) if ARQ_EXPERIMENTOS.exists() else pd.DataFrame()
    if len(tabela):
        tabela = tabela[tabela["id_experimento"] != registro["id_experimento"]]
    tabela = pd.concat([tabela, pd.DataFrame([registro])], ignore_index=True)
    tabela.to_csv(ARQ_EXPERIMENTOS, index=False)


def atualizar_conclusao(id_experimento, texto):
    tabela = pd.read_csv(ARQ_EXPERIMENTOS, dtype={"conclusao": "object"})
    tabela.loc[tabela["id_experimento"] == id_experimento, "conclusao"] = texto
    tabela.to_csv(ARQ_EXPERIMENTOS, index=False)


def registro_base(id_experimento, hipotese, colunas, modelo, parametros, conjunto, preparacao=PREPARACAO_PADRAO):
    return {"id_experimento": id_experimento, "data_hora": datetime.now().isoformat(timespec="seconds"),
            "alvo": ALVO, "hipotese": hipotese, "features": "|".join(colunas), "n_features_entrada": len(colunas),
            "preparacao": preparacao, "modelo": modelo, "parametros": json.dumps(parametros, default=str),
            "conjunto_avaliado": conjunto, "semente": SEMENTE}


def avaliar_cv(id_experimento, hipotese, colunas, regressor, parametros=None):
    """Valida o pipeline completo nas 3 dobras fixas e grava o resultado em experimentos.csv."""
    pipeline = montar_pipeline(colunas, regressor)
    inicio = time.perf_counter()
    res = cross_validate(pipeline, dev[colunas], y_dev, cv=DOBRAS, scoring=METRICAS,
                         return_train_score=True, n_jobs=1, error_score="raise")
    tempo_total = time.perf_counter() - inicio

    registro = registro_base(id_experimento, hipotese, colunas, type(regressor).__name__,
                             parametros if parametros is not None else regressor.get_params(),
                             "validacao_cruzada_desenvolvimento_3_dobras")
    for metrica in METRICAS:
        sinal = 1 if metrica == "r2" else -1
        validacao, treino = sinal * res[f"test_{metrica}"], sinal * res[f"train_{metrica}"]
        for k, valor in enumerate(validacao, start=1):
            registro[f"{metrica}_dobra{k}"] = valor
        registro[f"{metrica}_media"] = validacao.mean()
        registro[f"{metrica}_dp"] = validacao.std(ddof=1)
        registro[f"{metrica}_treino_media"] = treino.mean()
    registro["tempo_ajuste_medio_s"] = res["fit_time"].mean()
    registro["tempo_total_s"] = tempo_total
    registro["conclusao"] = ""
    salvar_experimento(registro)
    print(f"{id_experimento}: RMSE = {registro['rmse_media']:.2f} ± {registro['rmse_dp']:.2f} | "
          f"MAE = {registro['mae_media']:.2f} | R² = {registro['r2_media']:.4f} | {tempo_total:.0f} s")
    return registro


resultados = {}

## 9. Validação cruzada: baseline e regressão linear

| ID | Hipótese |
|---|---|
| `LC-LIN-00` | Baseline: prever a média do treino da dobra para todos. Referência de erro sem usar features. |
| `LC-LIN-01` | MQO com as **features originais** (sem `IN_TREINEIRO`) já reduz o erro em relação à média. |
| `LC-LIN-02` | Acrescentar as **3 features derivadas** (renda per capita, escolaridade máxima, perfil digital) reduz o erro do MQO. |
| `LC-LIN-03` | Ridge (`alpha=1`, configuração inicial) com o conjunto completo: com milhões de linhas e ~70 colunas, a regularização deve ter efeito desprezível. |

In [ ]:
resultados["LC-LIN-00"] = avaliar_cv(
    "LC-LIN-00", "Baseline pela média do treino de cada dobra; não usa features.",
    X_COMPLETO, DummyRegressor(strategy="mean"))

resultados["LC-LIN-01"] = avaliar_cv(
    "LC-LIN-01", "MQO com as features originais reduz o erro em relação ao baseline.",
    X_ORIGINAIS, LinearRegression())

resultados["LC-LIN-02"] = avaliar_cv(
    "LC-LIN-02", "As 3 features derivadas reduzem o erro do MQO em relação às originais.",
    X_COMPLETO, LinearRegression())

resultados["LC-LIN-03"] = avaliar_cv(
    "LC-LIN-03", "Ridge alpha=1 (configuração inicial) tem desempenho igual ao MQO com n >> p.",
    X_COMPLETO, Ridge(alpha=1.0))

## 10. Sensibilidade do Ridge ao `alpha`

Busca com `GridSearchCV` em **10 valores** de `alpha` (escala log, de 10⁻² a 10⁶), nas mesmas 3 dobras, critério = menor RMSE médio.
Como as features estão padronizadas e cada treino tem ~1,5 milhão de linhas, só valores muito altos de `alpha` devem mudar o resultado —
a busca serve para mostrar isso com evidência, e já deixa a busca pronta caso a regressão linear fique entre os três melhores regressores.

In [ ]:
ALPHAS = np.logspace(-2, 6, 10)
busca_ridge = GridSearchCV(
    montar_pipeline(X_COMPLETO, Ridge()),
    param_grid={"regressor__alpha": ALPHAS},
    scoring=METRICAS, refit="rmse", cv=DOBRAS, n_jobs=1, return_train_score=True, error_score="raise")

inicio = time.perf_counter()
busca_ridge.fit(dev[X_COMPLETO], y_dev)
tempo_busca = time.perf_counter() - inicio

cv_ridge = pd.DataFrame(busca_ridge.cv_results_)
tabela_ridge = pd.DataFrame({
    "alpha": cv_ridge["param_regressor__alpha"].astype(float),
    "rmse_media": -cv_ridge["mean_test_rmse"], "rmse_dp": cv_ridge["std_test_rmse"],
    "mae_media": -cv_ridge["mean_test_mae"], "r2_media": cv_ridge["mean_test_r2"],
    "rmse_treino_media": -cv_ridge["mean_train_rmse"], "tempo_ajuste_medio_s": cv_ridge["mean_fit_time"],
})
tabela_ridge.to_csv(PASTA_RESULTADOS / "busca_ridge_lc.csv", index=False)
print(f"Busca concluída em {tempo_busca:.0f} s. Melhor alpha = {busca_ridge.best_params_['regressor__alpha']:.4g}")
display(tabela_ridge.style.format({"alpha": "{:.4g}", "rmse_media": "{:.3f}", "rmse_dp": "{:.3f}", "mae_media": "{:.3f}",
                                   "r2_media": "{:.5f}", "rmse_treino_media": "{:.3f}", "tempo_ajuste_medio_s": "{:.1f}"}))

# Registro do melhor alpha com as métricas por dobra (std do cv_results_ é populacional; recalculamos com ddof=1)
melhor = busca_ridge.best_index_
registro = registro_base("LC-LIN-04", "Busca de alpha do Ridge (10 valores, 1e-2 a 1e6) não melhora o MQO.",
                         X_COMPLETO, "Ridge", {"alpha": busca_ridge.best_params_["regressor__alpha"],
                                               "grade_alpha": ALPHAS.tolist()},
                         "validacao_cruzada_desenvolvimento_3_dobras")
for metrica in METRICAS:
    sinal = 1 if metrica == "r2" else -1
    por_dobra = np.array([sinal * cv_ridge.loc[melhor, f"split{k}_test_{metrica}"] for k in range(3)])
    for k, valor in enumerate(por_dobra, start=1):
        registro[f"{metrica}_dobra{k}"] = valor
    registro[f"{metrica}_media"], registro[f"{metrica}_dp"] = por_dobra.mean(), por_dobra.std(ddof=1)
    registro[f"{metrica}_treino_media"] = sinal * cv_ridge.loc[melhor, f"mean_train_{metrica}"]
registro["tempo_ajuste_medio_s"] = cv_ridge.loc[melhor, "mean_fit_time"]
registro["tempo_total_s"] = tempo_busca
registro["conclusao"] = ""
salvar_experimento(registro)
resultados["LC-LIN-04"] = registro

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.errorbar(tabela_ridge["alpha"], tabela_ridge["rmse_media"], yerr=tabela_ridge["rmse_dp"],
            color=COR_MODELO, lw=2, marker="o", ms=6, capsize=3, label="Ridge (validação, média ± dp)")
ax.axhline(resultados["LC-LIN-02"]["rmse_media"], color=COR_BASELINE, ls="--", lw=1.2, label="MQO sem regularização")
ax.set_xscale("log")
ax.set(title="RMSE do Ridge por alpha — validação cruzada (3 dobras), desenvolvimento",
       xlabel="alpha (escala log)", ylabel="RMSE (pontos)")
ax.legend(loc="upper left")
fig.text(0, -0.05, FONTE_DADOS, fontsize=8, color=COR_TEXTO_SEC)
fig.savefig(PASTA_FIGURAS / "lc_ridge_alpha_dev.png")
plt.show()

## 11. Comparação dos experimentos (validação cruzada)

In [ ]:
ordem = ["LC-LIN-00", "LC-LIN-01", "LC-LIN-02", "LC-LIN-03", "LC-LIN-04"]
rotulos = {"LC-LIN-00": "Baseline (média)", "LC-LIN-01": "MQO — originais", "LC-LIN-02": "MQO — completo",
           "LC-LIN-03": "Ridge α=1 — completo",
           "LC-LIN-04": f"Ridge α={busca_ridge.best_params_['regressor__alpha']:.3g} — completo"}
comparacao = pd.DataFrame([{
    "experimento": i, "modelo": rotulos[i],
    "RMSE": resultados[i]["rmse_media"], "RMSE dp": resultados[i]["rmse_dp"],
    "MAE": resultados[i]["mae_media"], "MAE dp": resultados[i]["mae_dp"],
    "R²": resultados[i]["r2_media"], "R² dp": resultados[i]["r2_dp"],
    "RMSE treino": resultados[i]["rmse_treino_media"],
    "ajuste médio (s)": resultados[i]["tempo_ajuste_medio_s"], "tempo total (s)": resultados[i]["tempo_total_s"],
} for i in ordem])
comparacao.to_csv(PASTA_RESULTADOS / "comparacao_regressao_linear_cv.csv", index=False)
display(comparacao.style.format({c: "{:.2f}" for c in comparacao.columns if c not in ("experimento", "modelo")}
                                | {"R²": "{:.4f}", "R² dp": "{:.4f}"}).hide(axis="index"))

fig, ax = plt.subplots(figsize=(8, 3.6))
cores = [COR_BASELINE] + [COR_MODELO] * (len(ordem) - 1)
posicoes = np.arange(len(ordem))[::-1]
ax.barh(posicoes, comparacao["RMSE"], xerr=comparacao["RMSE dp"], color=cores, height=0.6,
        error_kw=dict(ecolor=COR_TEXTO_SEC, capsize=3, lw=1))
for p, valor in zip(posicoes, comparacao["RMSE"]):
    ax.text(valor + 0.6, p, f"{valor:.2f}", va="center", color=COR_TEXTO_SEC, fontsize=9)
ax.set_yticks(posicoes, comparacao["modelo"])
ax.set_xlim(0, comparacao["RMSE"].max() * 1.12)
ax.grid(axis="y", visible=False)
ax.set(title="RMSE médio nas 3 dobras (± dp) — desenvolvimento", xlabel="RMSE (pontos de nota)")
fig.text(0, -0.05, FONTE_DADOS, fontsize=8, color=COR_TEXTO_SEC)
fig.savefig(PASTA_FIGURAS / "lc_comparacao_linear_cv.png")
plt.show()

In [ ]:
# Conclusões automáticas registradas em experimentos.csv (complemente o texto no artigo).
b, o, c = resultados["LC-LIN-00"], resultados["LC-LIN-01"], resultados["LC-LIN-02"]
r1, r4 = resultados["LC-LIN-03"], resultados["LC-LIN-04"]

def relativo(a, ref):
    return (ref - a) / ref

atualizar_conclusao("LC-LIN-00", f"Referência: RMSE {b['rmse_media']:.2f} (≈ desvio padrão da nota); R² ≈ 0 por construção.")
atualizar_conclusao("LC-LIN-01", f"RMSE {o['rmse_media']:.2f}, {relativo(o['rmse_media'], b['rmse_media']):.1%} menor que o baseline; "
                                 f"R² {o['r2_media']:.4f}.")
dif = o["rmse_media"] - c["rmse_media"]
atualizar_conclusao("LC-LIN-02", f"Derivadas mudam o RMSE em {-dif:+.3f} pontos vs. originais (dp entre dobras ≈ {c['rmse_dp']:.3f}); "
                                 f"{'ganho maior que a variação entre dobras' if dif > 2 * c['rmse_dp'] else 'diferença pequena frente à variação entre dobras'}.")
atualizar_conclusao("LC-LIN-03", f"Diferença para o MQO: {r1['rmse_media'] - c['rmse_media']:+.4f} pontos de RMSE.")
atualizar_conclusao("LC-LIN-04", f"Melhor alpha={busca_ridge.best_params_['regressor__alpha']:.4g}; diferença para o MQO: "
                                 f"{r4['rmse_media'] - c['rmse_media']:+.4f} pontos de RMSE.")

for i in ordem:
    print(i, "->", pd.read_csv(ARQ_EXPERIMENTOS).set_index("id_experimento").loc[i, "conclusao"])

**Como interpretar**

* Compare cada modelo com o **baseline**: o RMSE do baseline é praticamente o desvio padrão da nota.
* **RMSE de treino ≈ RMSE de validação** → não há sobreajuste; o modelo linear tem poucos parâmetros para milhões de linhas.
  Erro alto e próximo nos dois indica **subajuste e/ou falta de informação nas features** (o perfil socioeconômico não determina a nota individual) — um R² baixo, sozinho, não prova subajuste.
* **Ridge ≈ MQO**: com n ≫ p a penalização é irrelevante até valores de `alpha` muito altos; a partir daí ela só piora (encolhe coeficientes úteis).
* A escolha da "solução linear" segue a regra do enunciado: o mais simples entre os de desempenho equivalente na validação cruzada → **MQO com `X_COMPLETO`** (a menos que os resultados acima mostrem outra coisa).

## 12. Análise dos erros no desenvolvimento (previsões fora da dobra)

`cross_val_predict` nas mesmas 3 dobras gera, para cada participante do desenvolvimento, uma previsão feita por um modelo
que **não treinou nele**. Resíduo = nota observada − nota prevista.

As **faixas de nota** e os **grupos** são definidos aqui, no desenvolvimento, e serão mantidos na avaliação final do teste:

In [ ]:
SOLUCAO_LINEAR = ("LC-LIN-02", X_COMPLETO, LinearRegression())

FAIXAS_NOTA = [-0.001, 0.0, 350, 450, 550, 650, 1000]
ROTULOS_FAIXAS = ["0 (zero)", "(0, 350]", "(350, 450]", "(450, 550]", "(550, 650]", "> 650"]
ROTULOS_ESCOLA = {1: "Não respondeu", 2: "Pública", 3: "Privada"}

_, colunas_sol, regressor_sol = SOLUCAO_LINEAR
inicio = time.perf_counter()
previsao_oof = cross_val_predict(montar_pipeline(colunas_sol, regressor_sol), dev[colunas_sol], y_dev, cv=DOBRAS, n_jobs=1)
print(f"Previsões fora da dobra em {time.perf_counter() - inicio:.0f} s")

analise = pd.DataFrame({
    "observada": y_dev, "prevista": previsao_oof, "residuo": y_dev - previsao_oof,
    "faixa_nota": pd.cut(y_dev, FAIXAS_NOTA, labels=ROTULOS_FAIXAS),
    "tipo_escola": dev["TP_ESCOLA"].map(ROTULOS_ESCOLA), "uf": dev["SG_UF_PROVA"],
})


def erros_por_grupo(tabela, coluna):
    return (tabela.groupby(coluna, observed=True)
            .apply(lambda g: pd.Series({
                "participantes": len(g),
                "RMSE": np.sqrt(np.mean(g["residuo"] ** 2)),
                "MAE": np.mean(np.abs(g["residuo"])),
                "residuo_medio": g["residuo"].mean(),
                "nota_media_obs": g["observada"].mean(),
                "nota_media_prev": g["prevista"].mean()}), include_groups=False)
            .astype({"participantes": int}))


print(f"Geral (OOF): RMSE = {np.sqrt(mean_squared_error(y_dev, previsao_oof)):.2f} | "
      f"MAE = {mean_absolute_error(y_dev, previsao_oof):.2f} | R² = {r2_score(y_dev, previsao_oof):.4f}")
print(f"Amplitude das previsões: {previsao_oof.min():.1f} a {previsao_oof.max():.1f} "
      f"(notas observadas: {y_dev.min():.1f} a {y_dev.max():.1f})")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.8))
hb = ax1.hexbin(analise["observada"], analise["prevista"], gridsize=70, bins="log", cmap="Blues", mincnt=1)
ax1.plot([0, 850], [0, 850], color=COR_TREINO, lw=1.5, ls="--", label="previsão perfeita")
ax1.set(xlim=(0, 850), ylim=(0, 850), xlabel="Nota observada", ylabel="Nota prevista",
        title="Observada × prevista — MQO, desenvolvimento (fora da dobra)")
ax1.legend(loc="upper left")
fig.colorbar(hb, ax=ax1, label="participantes (escala log)")

hb2 = ax2.hexbin(analise["prevista"], analise["residuo"], gridsize=70, bins="log", cmap="Blues", mincnt=1)
ax2.axhline(0, color=COR_TREINO, lw=1.5, ls="--")
ax2.set(xlabel="Nota prevista", ylabel="Resíduo (observada − prevista)", title="Resíduos × previsão")
fig.colorbar(hb2, ax=ax2, label="participantes (escala log)")
fig.text(0, -0.04, FONTE_DADOS + " Agregação em células hexagonais com todos os participantes do desenvolvimento.",
         fontsize=8, color=COR_TEXTO_SEC)
fig.savefig(PASTA_FIGURAS / "lc_linear_residuos_dev.png")
plt.show()

In [ ]:
por_faixa = erros_por_grupo(analise, "faixa_nota")
por_escola = erros_por_grupo(analise, "tipo_escola")
por_uf = erros_por_grupo(analise, "uf").sort_values("RMSE")
por_faixa.to_csv(PASTA_RESULTADOS / "lc_linear_erros_por_faixa_dev.csv")
por_escola.to_csv(PASTA_RESULTADOS / "lc_linear_erros_por_escola_dev.csv")
por_uf.to_csv(PASTA_RESULTADOS / "lc_linear_erros_por_uf_dev.csv")

formato = {"participantes": "{:,.0f}", "RMSE": "{:.1f}", "MAE": "{:.1f}", "residuo_medio": "{:+.1f}",
           "nota_media_obs": "{:.1f}", "nota_media_prev": "{:.1f}"}
display(por_faixa.style.format(formato).set_caption("Erros por faixa de nota (desenvolvimento, fora da dobra)"))
display(por_escola.style.format(formato).set_caption("Erros por tipo de escola (desenvolvimento, fora da dobra)"))

fig, eixos = plt.subplots(1, 3, figsize=(18, 4.6), gridspec_kw={"width_ratios": [1.7, 0.9, 2]})
for ax, tabela, titulo in [(eixos[0], por_faixa, "Por faixa de nota"), (eixos[1], por_escola, "Por tipo de escola"),
                           (eixos[2], por_uf, "Por UF de prova")]:
    x = np.arange(len(tabela))
    ax.bar(x, tabela["RMSE"], color=COR_MODELO, width=0.7)
    ax.set_xticks(x, [f"{i}\nn = " + f"{n:,}".replace(",", ".") if titulo != "Por UF de prova" else i
                      for i, n in zip(tabela.index, tabela["participantes"])],
                  fontsize=7.5, rotation=0 if titulo != "Por UF de prova" else 90)
    ax.axhline(np.sqrt(np.mean(analise["residuo"] ** 2)), color=COR_BASELINE, ls="--", lw=1)
    ax.grid(axis="x", visible=False)
    ax.set_title(titulo, fontsize=10)
eixos[0].set_ylabel("RMSE (pontos)")
fig.suptitle("RMSE do MQO por grupo — desenvolvimento, fora da dobra (tracejado = RMSE geral)", fontweight="bold", y=1.02)
fig.text(0, -0.1, FONTE_DADOS + " Tamanhos de cada UF em lc_linear_erros_por_uf_dev.csv.", fontsize=8, color=COR_TEXTO_SEC)
fig.savefig(PASTA_FIGURAS / "lc_linear_erros_grupos_dev.png")
plt.show()

**O que observar:** um modelo linear sobre o perfil tende a prever valores próximos da média
(amplitude das previsões bem menor que a das notas). Por isso o resíduo médio costuma ser **positivo nas faixas altas**
(o modelo subestima) e **negativo nas baixas** (superestima), com os maiores erros nas notas **zero** e **> 650** — faixas raras.
Isso motiva os experimentos de **balanceamento** e de transformação do alvo previstos no enunciado.

## 13. Coeficientes do MQO ajustado no desenvolvimento completo

Pelo protocolo (passo 4), o baseline e a regressão linear sem regularização são reajustados nos 80% de desenvolvimento.
Os coeficientes abaixo vêm desse ajuste. **Leitura:** para variáveis padronizadas, pontos de nota por +1 desvio padrão;
para one-hot, diferença em pontos em relação à categoria de referência, mantendo as demais constantes.
São **associações**, não efeitos causais, e não descrevem a capacidade individual de nenhum participante.

In [ ]:
inicio = time.perf_counter()
modelo_mqo_dev = montar_pipeline(X_COMPLETO, LinearRegression()).fit(dev[X_COMPLETO], y_dev)
baseline_dev = montar_pipeline(X_COMPLETO, DummyRegressor(strategy="mean")).fit(dev[X_COMPLETO], y_dev)
print(f"Ajuste no desenvolvimento completo: {time.perf_counter() - inicio:.0f} s | "
      f"intercepto = {modelo_mqo_dev['regressor'].intercept_:.1f} | média do baseline = {baseline_dev['regressor'].constant_[0][0]:.1f}")

coeficientes = pd.Series(modelo_mqo_dev["regressor"].coef_,
                         index=modelo_mqo_dev["preparacao"].get_feature_names_out()).rename("coeficiente")
coeficientes.sort_values().to_csv(PASTA_RESULTADOS / "lc_linear_coeficientes.csv", header=True)

maiores = coeficientes.reindex(coeficientes.abs().sort_values(ascending=False).index[:20]).sort_values()
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(range(len(maiores)), maiores, color=[COR_MODELO if v > 0 else COR_TREINO for v in maiores], height=0.7)
ax.set_yticks(range(len(maiores)), [n.split("__", 1)[1] for n in maiores.index], fontsize=8)
ax.axvline(0, color=COR_TEXTO_SEC, lw=0.8)
ax.grid(axis="y", visible=False)
ax.set(title="20 maiores coeficientes (em módulo) — MQO, desenvolvimento", xlabel="Pontos de nota")
fig.text(0, -0.04, FONTE_DADOS + " Azul: associação positiva; laranja: negativa.", fontsize=8, color=COR_TEXTO_SEC)
fig.savefig(PASTA_FIGURAS / "lc_linear_coeficientes.png")
plt.show()

## 14. Curva de aprendizado (diagnóstico de sub/sobreajuste)

Mesmo pipeline e mesma configuração (MQO, `X_COMPLETO`), treinado com 10%, 25%, 50%, 75% e 100% das linhas de treino
**de cada dobra**, com a validação de cada dobra fixa e subconjuntos aleatórios com semente 42. Não usa o teste.
(A curva obrigatória do artigo é a do **modelo final**; esta serve para discutir a regressão linear.)

In [ ]:
TAMANHOS = [0.10, 0.25, 0.50, 0.75, 1.00]
inicio = time.perf_counter()
n_treino, rmse_treino, rmse_valid = learning_curve(
    montar_pipeline(X_COMPLETO, LinearRegression()), dev[X_COMPLETO], y_dev,
    train_sizes=TAMANHOS, cv=DOBRAS, scoring="neg_root_mean_squared_error",
    shuffle=True, random_state=SEMENTE, n_jobs=1, error_score="raise")
rmse_treino, rmse_valid = -rmse_treino, -rmse_valid
print(f"Curva de aprendizado em {time.perf_counter() - inicio:.0f} s")

curva = pd.DataFrame({"fracao_treino_dobra": TAMANHOS, "n_treino": n_treino,
                      "rmse_treino_media": rmse_treino.mean(1), "rmse_treino_dp": rmse_treino.std(1, ddof=1),
                      "rmse_validacao_media": rmse_valid.mean(1), "rmse_validacao_dp": rmse_valid.std(1, ddof=1)})
curva.to_csv(PASTA_RESULTADOS / "lc_linear_curva_aprendizado.csv", index=False)
display(curva.style.format({"n_treino": "{:,.0f}", "fracao_treino_dobra": "{:.0%}"} |
                           {c: "{:.3f}" for c in curva.columns if c.startswith("rmse")}).hide(axis="index"))

fig, ax = plt.subplots(figsize=(8, 4))
for media, dp, cor, rotulo in [(curva["rmse_treino_media"], curva["rmse_treino_dp"], COR_TREINO, "Treino"),
                               (curva["rmse_validacao_media"], curva["rmse_validacao_dp"], COR_MODELO, "Validação")]:
    ax.plot(n_treino, media, color=cor, lw=2, marker="o", ms=6, label=rotulo)
    ax.fill_between(n_treino, media - dp, media + dp, color=cor, alpha=0.18, lw=0)
ax.axhline(resultados["LC-LIN-00"]["rmse_media"], color=COR_BASELINE, ls="--", lw=1, label="Baseline (média)")
ax.xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v/1000:,.0f} mil".replace(",", ".")))
ax.set(title="Curva de aprendizado — MQO, validação cruzada (3 dobras), desenvolvimento",
       xlabel="Participantes no treino da dobra", ylabel="RMSE (média ± dp)")
ax.legend()
fig.text(0, -0.05, FONTE_DADOS, fontsize=8, color=COR_TEXTO_SEC)
fig.savefig(PASTA_FIGURAS / "lc_linear_curva_aprendizado.png")
plt.show()

**Leitura esperada:** curvas de treino e validação praticamente coladas e estáveis desde 10% → **sem sobreajuste**, e
**mais dados não reduzem o erro** do modelo linear. O erro alto e próximo nos dois indica que o limite está na
**capacidade do modelo** (relações não lineares e interações não modeladas) e/ou na **informação das features** — o que
se investiga comparando com modelos não lineares (árvores, boosting) nas mesmas dobras.

## 15. Avaliação no teste reservado — **desligada**

O enunciado proíbe avaliar o teste no acompanhamento e antes de fixar a solução final.
Quando a equipe tiver escolhido a solução final **apenas pela validação cruzada**, mude `AVALIAR_TESTE = True`.
A célula reajusta baseline e MQO nos 80% de desenvolvimento (já feito na Seção 13), prevê o teste e grava:

* `resultados/previsoes_teste.csv` — `NU_INSCRICAO`, nota observada e uma coluna por modelo (preserva colunas de outros modelos já gravadas);
* `resultados/experimentos.csv` — linhas `LC-LIN-T00` (baseline) e `LC-LIN-T02` (MQO) com `conjunto_avaliado = teste`.

Depois disso, **não volte a alterar features, técnicas ou hiperparâmetros** com base no teste.

In [ ]:
AVALIAR_TESTE = False

if AVALIAR_TESTE:
    y_teste = teste[ALVO].to_numpy()
    previsoes = {}
    for id_exp, nome, modelo in [("LC-LIN-T00", "baseline_media", baseline_dev),
                                 ("LC-LIN-T02", "regressao_linear_mqo", modelo_mqo_dev)]:
        inicio = time.perf_counter()
        previsto = modelo.predict(teste[X_COMPLETO])
        tempo = time.perf_counter() - inicio
        previsoes[f"prev_{nome}"] = previsto
        registro = registro_base(id_exp, "Avaliação final no teste reservado (modelo ajustado nos 80% de desenvolvimento).",
                                 X_COMPLETO, type(modelo["regressor"]).__name__, modelo["regressor"].get_params(), "teste")
        registro.update({"rmse_media": np.sqrt(mean_squared_error(y_teste, previsto)),
                         "mae_media": mean_absolute_error(y_teste, previsto),
                         "r2_media": r2_score(y_teste, previsto), "tempo_total_s": tempo, "conclusao": ""})
        salvar_experimento(registro)
        print(f"{id_exp} {nome}: RMSE = {registro['rmse_media']:.2f} | MAE = {registro['mae_media']:.2f} | R² = {registro['r2_media']:.4f}")

    tabela_prev = pd.DataFrame({COLUNA_ID: teste[COLUNA_ID], f"{ALVO}_observada": y_teste, **previsoes})
    if ARQ_PREVISOES_TESTE.exists():
        anteriores = pd.read_csv(ARQ_PREVISOES_TESTE, dtype={COLUNA_ID: str})
        manter = [c for c in anteriores.columns if c not in tabela_prev.columns]
        tabela_prev = tabela_prev.merge(anteriores[[COLUNA_ID] + manter], on=COLUNA_ID, how="left", validate="one_to_one")
    tabela_prev.to_csv(ARQ_PREVISOES_TESTE, index=False)
    print("Previsões gravadas em", ARQ_PREVISOES_TESTE)
else:
    print("Teste NÃO avaliado (AVALIAR_TESTE = False). Participantes reservados:", f"{len(teste):,}".replace(",", "."))

## 16. Arquivos gerados

In [ ]:
gerados = sorted([*PASTA_SPLITS.glob("*"), *PASTA_RESULTADOS.glob("*.csv"), *PASTA_FIGURAS.glob("lc_*.png")])
for caminho in gerados:
    print(f"{caminho.stat().st_size / 1e6:8.2f} MB  {caminho.relative_to(PASTA_PROJETO)}")

if NO_COLAB:
    # As saídas estão em /content e somem ao desconectar: baixa um .zip (sem a base, que você já tem).
    import zipfile
    pacote = Path("/content/saidas_regressao_linear_lc.zip")
    with zipfile.ZipFile(pacote, "w", zipfile.ZIP_DEFLATED) as zf:
        for caminho in gerados:
            zf.write(caminho, caminho.relative_to(PASTA_PROJETO))
    print("Baixando", pacote.name, "— descompacte na raiz do projeto.")
    files.download(str(pacote))

## Próximos passos sugeridos

* Rodar os demais regressores obrigatórios **reutilizando `dados/splits/particoes_lc.parquet`** e o mesmo `experimentos.csv`.
* Hipótese de aprofundamento barata com este mesmo notebook: **one-hot vs. ordinal** para `Q006`/`Q001`/`Q002`
  (a relação renda × nota é monotônica mas côncava — o one-hot deixa o modelo linear capturar a curvatura).
* Balanceamento (pesos por faixa de nota definidos no treino de cada dobra — `LinearRegression` aceita `sample_weight`)
  para atacar o erro nas faixas extremas vistas na Seção 12.
* PCA sobre o bloco codificado + padronizado, comparado ao MQO sem redução.